Use this notebook to extract GTFS Data from 511 to get realtime vehicle locations and speeds.

In [23]:
import pandas as pd
import geopandas as gpd
import requests
from google.transit import gtfs_realtime_pb2
from tqdm import tqdm
import time
from datetime import datetime
from shapely.geometry import Point
import plotly.express as px

In [ ]:
# 511 API
api_key = '' #INSERT YOUR 511 API KEY HERE
url = f'https://api.511.org/Transit/VehiclePositions?api_key={api_key}&agency=SC'

columns = ['bus_id', 'trip_id', 'route_id', 'lat', 'lon', 'time', 'speed', 'direction']
selected_routes = ['22', 'Rapid 522', '23', 'Rapid 523', '25', '60']

def fetch_realtime_positions_speeds(api_url, iterations=1, delay=60, existing_bus_data=pd.DataFrame(columns=columns)):
    '''
    Uses the provided 511 API key and to query live VTA vehicle location data repeatedly on a given delay, up to 
    a provided number of iterations. The response is then filtered for vehicles on bus routes 22, Rapid 522, 23, 
    Rapid 523, 25, and 60, for their position, travel direction, speed, and timestamp, and appended to the existing
    list of vehicle locations. Finally, it exports the data to a csv file with a timestamped file name for storage 
    and future cleaning. 

    Parameters:
        api_url (str): The API key and URL used to access live data from VTA and GTFS
        iterations (int): The number of times to query data.
        delay (int): The amount of time between consecutive queries of live data, in seconds.
        existing_bus_data (pd.DataFrame): Allows the option to insert an existing DataFrame to append data rather 
            than creating a new DataFrame

    Returns:
        existing_bus_data (pd.DataFrame): The DataFrame after all bus location data from GTFS has been queried.
            This data is also exported to a .csv file with a timestamped file name.
    '''

    count = 0
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"../data/raw/vehicle_route_location_speed_data_{timestamp}.csv"

    try:
        #set progress bar for quick progress checking for long queries
        with tqdm(total=iterations) as progress_bar:
            while count < iterations:
                response = requests.get(api_url)

                #if successful query
                if response.status_code == 200:
                    
                    try:
                        #Get GTFS response
                        feed = gtfs_realtime_pb2.FeedMessage()
                        feed.ParseFromString(response.content)

                        data_list = []

                        #For each vehicle
                        for entity in feed.entity:
                            if entity.HasField('vehicle'):
                                vehicle = entity.vehicle
                                trip = vehicle.trip
                                route_id = trip.route_id

                                #Check if the route_id is in the list of desired route IDs
                                if route_id in selected_routes:

                                    #Extract the vehicle's details
                                    vehicle_data = {
                                        'bus_id': entity.id,
                                        'trip_id': trip.trip_id,
                                        'route_id': route_id,
                                        'lat': vehicle.position.latitude,
                                        'lon': vehicle.position.longitude,
                                        'time': vehicle.timestamp,
                                        'speed': vehicle.position.speed,
                                        'direction' : trip.direction_id
                                    }

                                    data_list.append(vehicle_data)
                        
                        #append newly extracted data to existing DataFrame
                        new_bus_data = pd.DataFrame(data_list, columns=existing_bus_data.columns)
                        existing_bus_data = pd.concat([existing_bus_data, new_bus_data], ignore_index=True)

                    #if error for some reason, continue
                    except Exception as e:
                        print(e)
                    
                else:
                    print(f"Error: {response.status_code} - {response.text}")
                
                #delay to wait for live GTFS data to update
                #No delay after the last iteration
                if count < iterations - 1:
                    time.sleep(delay)
                
                count += 1
                progress_bar.update(1)

        #export final output
        existing_bus_data.to_csv(file_path, index=False)

    #if error for some reason, simply save existing data
    except:
        existing_bus_data.to_csv(file_path, index=False)

    return existing_bus_data

In [3]:
#Interupting this probably forces a kernel restart if running many iterations
#1 hr worth of data = iterations*delay/3600
#use delay >= 10 reflect how often the positions are refreshed

#this was originally run with iterations=960 and delay=30 for 8 hours of data
bus_data = fetch_realtime_positions_speeds(url, iterations=2, delay=30)
bus_data.drop_duplicates()

  0%|          | 0/2 [00:00<?, ?it/s]C:\Users\thele\AppData\Local\Temp\ipykernel_35748\2227748577.py:74: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  existing_bus_data = pd.concat([existing_bus_data, new_bus_data], ignore_index=True)
100%|██████████| 2/2 [00:30<00:00, 15.15s/it]


,bus_id,trip_id,route_id,lat,lon,time,speed,direction
0,0130,3926036,60,37.356800,-121.935059,1774994429,0.004116,0
1,0132,3926034,60,37.409344,-121.894775,1774994429,12.888781,0
2,0134,3926099,60,37.351013,-121.944382,1774994429,11.001814,1
3,0135,3926038,60,37.298889,-121.949829,1774994429,15.228453,0
4,0136,3926101,60,37.370144,-121.916084,1774994429,6.960888,1
...,...,...,...,...,...,...,...,...
159,8336,3916748,22,37.350937,-121.859314,1774994462,9.438945,0
160,8339,3916767,22,37.443573,-122.165573,1774994462,4.432930,0
161,8340,3916770,22,37.444122,-122.166161,1774994462,0.002058,0
162,8341,3916839,22,37.327461,-121.810715,1774994462,0.002572,1


In [4]:
bus_gdf_geometry = [Point(xy) for xy in zip(bus_data["lon"], bus_data["lat"])]

In [5]:
#convert data to local crs
bus_gdf = gpd.GeoDataFrame(bus_data, geometry=bus_gdf_geometry, crs="EPSG:4326")
bus_gdf = bus_gdf.to_crs("EPSG:2227")
bus_gdf

,bus_id,trip_id,route_id,lat,lon,time,speed,direction,geometry
0,0130,3926036,60,37.356800,-121.935059,1774994429,0.004116,0,POINT (6144585.967 1955583.701)
1,0132,3926034,60,37.409344,-121.894775,1774994429,12.888781,0,POINT (6156577.93 1974535.906)
2,0134,3926099,60,37.351013,-121.944382,1774994429,11.001814,1,POINT (6141844.028 1953518.61)
3,0135,3926038,60,37.298889,-121.949829,1774994429,15.228453,0,POINT (6139966.857 1934566.892)
4,0136,3926101,60,37.370144,-121.916084,1774994429,6.960888,1,POINT (6150173.672 1960357.668)
...,...,...,...,...,...,...,...,...,...
159,8336,3916748,22,37.350937,-121.859314,1774994462,9.438945,0,POINT (6166567.535 1953120.473)
160,8339,3916767,22,37.443573,-122.165573,1774994462,4.432930,0,POINT (6078158.582 1988282.713)
161,8340,3916770,22,37.444122,-122.166161,1774994462,0.002058,0,POINT (6077991.621 1988485.726)
162,8341,3916839,22,37.327461,-121.810715,1774994462,0.002572,1,POINT (6180572.796 1944372.251)


In [6]:
#using this for quick plotly example viz
bus_gdf_4326 = bus_gdf.copy().to_crs("EPSG:4326")

fig = px.scatter_mapbox(
    bus_gdf_4326,
    lat=bus_gdf_4326.geometry.y,
    lon=bus_gdf_4326.geometry.x,
    hover_data=["bus_id", "speed", "route_id"],
    color=bus_gdf_4326["speed"],
    opacity=0.25,
    zoom=10,
    height=600
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin={"r":0,"t":0,"l":0,"b":0}
)

#fig.write_html("Sample_Speed_Vis.html")
fig.show()

C:\Users\thele\AppData\Local\Temp\ipykernel_35748\734988329.py:4: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
